In [ ]:
!python -m pip install -r requirements.txt


/content
Cloning into 'UNLV-Research'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (360/360), done.
remote: Compressing objects: 100% (249/249), done.
remote: Total 360 (delta 144), reused 321 (delta 105), pack-reused 0 (from 0)
Receiving objects: 100% (360/360), 348.56 KiB | 1.86 MiB/s, done.
Resolving deltas: 100% (144/144), done.
/content/UNLV-Research/Phase-1
total 356
drwxr-xr-x 5 root root  4096 Mar  3 21:50 .
drwxr-xr-x 6 root root  4096 Mar  3 21:50 ..
-rw-r--r-- 1 root root    55 Mar  3 21:50 00_run_phase1.bat
-rw-r--r-- 1 root root   170 Mar  3 21:50 00_run_phase1.py
-rw-r--r-- 1 root root   168 Mar  3 21:50 01_collect_khan_academy.py
-rw-r--r-- 1 root root   180 Mar  3 21:50 02_collect_tiny_textbooks.py
-rw-r--r-- 1 root root   179 Mar  3 21:50 03_extract_khan_taxonomy.py
-rw-r--r-- 1 root root   196 Mar  3 21:50 04_build_corpus_index.py
-rw-r--r-- 1 root root   152 Mar  3 21:50 05_compute_metrics.py
-rw-r--r-- 1 root root   145 Mar  3 21:50 06_b

# Phase-1 One-Run Pipeline (Colab + VSCode)

This notebook is fixed to run the full chain without per-step toggles:
`00 -> 08 -> 09 -> 10 -> 11 -> 06 -> 07`

If manual labels are missing, it stops before `09` with a clear message.


## 0) Runtime Setup

If auto-detection fails, set one of these and rerun this cell:

```python
%env PHASE1_REPO_DIR=/content/drive/MyDrive/<repo-root>
# or
%env PHASE1_PHASE1_DIR=/content/drive/MyDrive/<repo-root>/Phase-1
```


In [30]:
import os
import sys
import csv
import json
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)


def _run_capture(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        return ''
    return out.strip()


def _is_valid_phase1_dir(path_obj):
    p = Path(path_obj)
    return (
        p.exists()
        and p.is_dir()
        and (p / '00_run_phase1.py').exists()
        and (p / '05_compute_metrics.py').exists()
        and (p / '11_certify_phase1.py').exists()
    )


def _dedup(paths):
    out, seen = [], set()
    for p in paths:
        try:
            r = Path(p).resolve()
        except Exception:
            continue
        s = str(r)
        if s in seen:
            continue
        seen.add(s)
        out.append(r)
    return out


def resolve_repo_and_phase1():
    manual_phase1 = os.environ.get('PHASE1_PHASE1_DIR', '').strip()
    manual_repo = os.environ.get('PHASE1_REPO_DIR', '').strip()

    candidates = []
    if manual_phase1:
        candidates.append(Path(manual_phase1).expanduser())
    if manual_repo:
        repo = Path(manual_repo).expanduser()
        candidates.extend([repo / 'Phase-1', repo])

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd / 'Phase-1'])
    for parent in list(cwd.parents)[:8]:
        candidates.extend([parent, parent / 'Phase-1'])

    if IN_COLAB:
        candidates.extend([
            Path('/content/UNLV-Research/Phase-1'),
            Path('/content/UNLV-Research/UNLV-Research/Phase-1'),
            Path('/content/drive/MyDrive/UNLV-Research/Phase-1'),
            Path('/content/drive/MyDrive/UNLV-Research/UNLV-Research/Phase-1'),
            Path('/content/drive/MyDrive/unlv/UNLV-Research/Phase-1'),
            Path('/content/drive/MyDrive/University/unlv/UNLV-Research/Phase-1'),
        ])

    for cand in _dedup(candidates):
        if _is_valid_phase1_dir(cand):
            return cand.parent, cand

    hits = _run_capture("find /content/drive -type f -name '00_run_phase1.py' 2>/dev/null | head -n 20")
    for line in [x.strip() for x in hits.splitlines() if x.strip()]:
        phase1 = Path(line).resolve().parent
        if _is_valid_phase1_dir(phase1):
            return phase1.parent, phase1

    raise FileNotFoundError(
        "Cannot locate Phase-1 directory. Run:\n"
        "!find /content/drive -type f -name '00_run_phase1.py' 2>/dev/null | head -n 20\n"
        "Then set %env PHASE1_REPO_DIR=<repo-root> or %env PHASE1_PHASE1_DIR=<repo-root>/Phase-1"
    )


REPO_DIR, PHASE1_DIR = resolve_repo_and_phase1()
os.environ['PHASE1_REPO_DIR'] = str(REPO_DIR)
os.environ['PHASE1_PHASE1_DIR'] = str(PHASE1_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'
PYTHON = sys.executable

print('Python   :', PYTHON)
print('Repo     :', REPO_DIR)
print('Phase-1  :', PHASE1_DIR)


def run_py(script_name, *args):
    cmd = [PYTHON, '-u', str(PHASE1_DIR / script_name), *args]
    print('\n$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, cwd=str(PHASE1_DIR), text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{script_name} failed with exit code {r.returncode}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python   : /usr/bin/python3
Repo     : /content/UNLV-Research
Phase-1  : /content/UNLV-Research/Phase-1


## 1) Optional dependency install (fresh Colab only)


In [31]:
INSTALL_DEPS = False
if INSTALL_DEPS:
    req = PHASE1_DIR / 'requirements.txt'
    if req.exists():
        cmd = [PYTHON, '-m', 'pip', 'install', '-r', str(req)]
        print('$', ' '.join(cmd))
        subprocess.run(cmd, check=False)
    else:
        print('requirements.txt not found; skipped.')
else:
    print('Dependency install skipped.')


Dependency install skipped.


## 2) Fixed execution profile


In [32]:
profile = {
    'PHASE1_DEVICE': 'auto',
    'PHASE1_CUDA_DEVICE': '0',
    'PHASE1_USE_GPU': '1',
    'PHASE1_DOMAIN_BATCH_SIZE': '256',
    'PHASE1_MAX_BATCHES': '0',
    'PHASE1_INDEX_MAX_BATCHES': '0',
    'PHASE1_TFIDF_MAX_FEATURES': '3000',
    'PHASE1_BUILD_TFIDF_MATRIX': '0',
    'PHASE1_STORE_DOC_TEXTS': '1',
    'PHASE1_DOC_TEXT_BACKEND': 'sqlite',
    'PHASE1_DOC_TEXT_INSERT_BATCH': '2000',
    'PHASE1_DOC_TEXT_CACHE_LIMIT': '2500',
    'PHASE1_ENABLE_MINHASH': '1',
    'PHASE1_QUERY_CACHE_LIMIT': '1024',
    'PHASE1_NGRAM_CACHE_LIMIT': '2048',
    'PHASE1_SEMANTIC_CANDIDATE_LIMIT': '80',
    'PHASE1_NGRAM_CANDIDATE_LIMIT': '80',
    'PHASE1_SKIP_REDUNDANCY': '0',
    'PHASE1_SKIP_PERPLEXITY': '0',
    'PHASE1_DATASETS_CONFIG': 'datasets_config.json',
    'PHASE1_IDENTITY_CONFIG': 'configs/metric_identity_v1.json',
}
for k, v in profile.items():
    os.environ[k] = v
print('Profile loaded.')


Profile loaded.


## 3) One-click full run (00 -> 08 -> 09 -> 10 -> 11 -> 06 -> 07)

- This cell runs the full pipeline chain.
- If labels are missing, it stops before `09`.


In [33]:
core_outputs = [
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
    PHASE1_DIR / 'outputs' / 'khan_analysis.jsonl',
    PHASE1_DIR / 'outputs' / 'tiny_textbooks_analysis.jsonl',
]
if all(p.exists() for p in core_outputs):
    print('[SKIP] 00_run_phase1.py (core outputs already exist)')
else:
    print('[RUN] 00_run_phase1.py (core outputs missing)')
    run_py('00_run_phase1.py')

labels_dir = PHASE1_DIR / 'validation' / 'labels'
checks = {
    'domain_labels_v1.csv': ('gold_primary', 'main_eval'),
    'quality_labels_v1.csv': ('gold_has_examples', 'main_eval'),
    'difficulty_sanity_v1.csv': ('gold_valid', 'main_eval'),
    'ood_labels_v1.csv': ('gold_in_domain', 'calibration'),
}

def non_empty(v):
    return str(v or '').strip() != ''

def label_status():
    missing = []
    detail = []
    for name, (col, split_name) in checks.items():
        p = labels_dir / name
        if not p.exists():
            missing.append(f'{name}: missing file')
            detail.append((name, split_name, 0, 0))
            continue
        with p.open('r', encoding='utf-8', newline='') as f:
            rows = list(csv.DictReader(f))
        split_rows = [r for r in rows if str(r.get('split', '')).strip() == split_name]
        labeled = [r for r in split_rows if non_empty(r.get(col))]
        detail.append((name, split_name, len(split_rows), len(labeled)))
        if len(split_rows) == 0 or len(labeled) == 0:
            missing.append(f'{name}: split={split_name} labeled=0')
    return missing, detail

# only generate templates when labels are not ready
missing, detail = label_status()
for name, split_name, total, labeled in detail:
    print(f'[LABEL] {name} split={split_name} total={total} labeled={labeled}')

if missing:
    print('\nLabels are incomplete. Generating templates via 08...')
    run_py('08_generate_label_templates.py')
    missing, detail = label_status()
    for name, split_name, total, labeled in detail:
        print(f'[LABEL-AFTER-08] {name} split={split_name} total={total} labeled={labeled}')

if missing:
    print('\nManual labeling required before 09:')
    for m in missing:
        print(' -', m)
    raise SystemExit('Stop here. Fill labels, then rerun this cell. 08 will be skipped if labels are complete.')

# 09~11: metric identity certification flow
run_py('09_calibrate_ood_thresholds.py')
run_py('10_score_metric_gates.py')
run_py('11_certify_phase1.py')

# final refresh
run_py('06_build_dashboard.py')
run_py('07_validate_phase1_outputs.py', '--require-gates', '--write-report', 'outputs/validation/final_validation_report.json')

print('\nFull chain completed.')


[RUN] 00_run_phase1.py (core outputs missing)

$ /usr/bin/python3 /content/UNLV-Research/Phase-1/00_run_phase1.py


RuntimeError: 00_run_phase1.py failed with exit code 1

## 4) Final output checkpoint


In [ ]:
outs = [
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
    PHASE1_DIR / 'outputs' / 'dashboard.html',
    PHASE1_DIR / 'outputs' / 'validation' / 'ood_calibration_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_main.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_transfer.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'full_validation_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'final_validation_report.json',
]
for p in outs:
    print(f"[{'OK' if p.exists() else 'MISS'}] {p}")

manifest_path = PHASE1_DIR / 'outputs' / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('
certification_status:', manifest.get('certification_status'))
    print('certification_failed_gates:', manifest.get('certification_failed_gates'))
